In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
# 1. 데이터 불러오기
url = "https://raw.githubusercontent.com/adlnlp/K-MHaS/refs/heads/main/data/kmhas_train.txt"
df = pd.read_csv(url, sep="\t")

In [ ]:
# 2. 이진 라벨 생성: 0~7 → 혐오(1), 8 → 비혐오(0)
df["hate_label"] = df["label"].apply(lambda x: 0 if x == 8 else 1)

In [ ]:
df["label"] = df["label"].apply(lambda x: int(str(x).split(",")[0]))

num_labels = 9  # 0~8
label2id = {i:i for i in range(num_labels)}
id2label = {i:str(i) for i in range(num_labels)}

print(df.head())


In [ ]:
# 3. train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["document"], df["hate_label"], test_size=0.1, random_state=42
)

In [ ]:
# 4. 토크나이저
tokenizer = BertTokenizer.from_pretrained("klue/bert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

In [ ]:
# 5. Dataset 클래스 정의
class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_len)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
# 6. Dataset, DataLoader 생성
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
# 7. BERT 모델 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2)
model.to(device)

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# 8. Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
# 9. 학습 루프
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 4443/4443 [24:59<00:00,  2.96it/s]


Epoch 1 - Average Loss: 0.0004


Epoch 2: 100%|██████████| 4443/4443 [24:58<00:00,  2.97it/s]


Epoch 2 - Average Loss: 0.0000


Epoch 3: 100%|██████████| 4443/4443 [24:57<00:00,  2.97it/s]

Epoch 3 - Average Loss: 0.0000


In [ ]:
 # 10. 검증
def predict_hate(texts, model, tokenizer, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(F.softmax(logits, dim=1), dim=1)

            preds.extend(predictions.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    print(f"Validation Accuracy: {acc:.4f}")

In [ ]:
# 11. 테스트용 문장 예측 함수 정의 및 실행
def predict_hate(texts, model, tokenizer, device):
    model.eval()
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

    return preds.cpu().tolist(), probs.cpu().tolist()

# 예시 문장
test_sentences = [
    "저 사람은 진짜 이상하다. 없어졌으면 좋겠다.",  # 혐오
    "오늘 날씨 너무 좋다. 산책 가고 싶다!",         # 비혐오
    "정치인들은 다 똑같아. 믿을 수 없어.",         # 혐오
    "꺼져라 ."         # 비혐오
]

# 예측
predictions, probabilities = predict_hate(test_sentences, model, tokenizer, device)

# 결과 출력
for text, pred, prob in zip(test_sentences, predictions, probabilities):
    label = "혐오" if pred == 1 else "비혐오"
    print(f"[{label}] {text} (확률: 혐오={prob[1]:.2f}, 비혐오={prob[0]:.2f})")


[혐오] 저 사람은 진짜 이상하다. 없어졌으면 좋겠다. (확률: 혐오=1.00, 비혐오=0.00)
[혐오] 오늘 날씨 너무 좋다. 산책 가고 싶다! (확률: 혐오=1.00, 비혐오=0.00)
[혐오] 정치인들은 다 똑같아. 믿을 수 없어. (확률: 혐오=1.00, 비혐오=0.00)
[혐오] 꺼져라 . (확률: 혐오=1.00, 비혐오=0.00)
